In [1]:
import json
import re
import os
from PIL import Image
import pdfplumber
import torch
import cv2
import numpy as np
# from pipe_fn import pipe

from transformers import pipeline
# =========================
# CROP PT → REFERENCE
# =========================
from pypdf import PdfReader, PdfWriter

from output_utils import save_split_output

from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)
# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)




def rotate_and_crop_pdf(
    pdf_path,
    output_dir="Moda_output_final"
):

    # =====================================
    # CREATE OUTPUT FOLDER
    # =====================================
    pdf_name = os.path.basename(pdf_path)
    pdf_dir = os.path.basename(pdf_path).split(".")[0].split("_")[-1]
    output_dir = os.path.join(output_dir, pdf_dir)
    os.makedirs(output_dir, exist_ok=True)

    # =====================================
    # CHECK IF ROTATION IS NEEDED
    # =====================================

    reader = PdfReader(pdf_path)
    rotate_remaining_pages = False

    if len(reader.pages) > 1:
        page2 = reader.pages[1]
        width = float(page2.mediabox.width)
        height = float(page2.mediabox.height)

        print(f"Page 2 Size -> Width={width}, Height={height}")

        if height > width:
            rotate_remaining_pages = True
            print(
                "📄 Page 2 is PORTRAIT."
                "\n🔄 Rotating pages from Page 2 onwards."
            )
        else:
            print(
                "📄 Page 2 is LANDSCAPE."
                "\n ⏭️ No rotation required."
            )

    # =====================================
    # ROTATE PDF
    # =====================================

    rotated_pdf_path = os.path.join(output_dir, f"rotated_{pdf_dir}.pdf")
    writer = PdfWriter()

    for page_num, page in enumerate(reader.pages, start=1):
        print(f"\n🚀 Processing Page {page_num}")

        if page_num > 1 and rotate_remaining_pages:
            print("🔄 Rotating page by 90°")
            page.rotate(90)
        else:
            print("⏭️ Keeping page as-is")

        writer.add_page(page)

    with open(rotated_pdf_path, "wb") as f:
        writer.write(f)

    print(f"\n✅ Rotated PDF Saved: {rotated_pdf_path}")

    # =====================================
    # OPEN ROTATED PDF
    # =====================================

    results = []

    pending_parts = []
    pending_page = None
    pending_table_idx = None
    pending_expected_rows = 0

    with pdfplumber.open(rotated_pdf_path) as pdf:

        for page_num, page in enumerate(pdf.pages, start=1):

            print(f"\n🧾 Cropping Page {page_num}")

            start_hits = sorted(
                page.search("Patient:"),
                key=lambda h: h["top"]
            )

            end_hits = sorted(
                page.search("Totals"),
                key=lambda h: h["bottom"]
            )

            start_positions = [
                hit["top"] - 10
                for hit in start_hits
            ]

            end_positions = [
                hit["bottom"] + 20
                for hit in end_hits
            ]

            table_counter_on_page = 0

            # =====================================
            # CLOSE CONTINUED TABLE
            # =====================================

            if pending_parts:

                if end_positions:

                    end_y = end_positions.pop(0)

                    bbox = (
                        0,
                        0,
                        page.width,
                        end_y
                    )

                    pending_parts.append(
                        page.crop(bbox)
                    )

                    pending_expected_rows += count_service_rows(
                        page,
                        0,
                        end_y
                    )



                    output_path = _merge_and_save(
                        pending_parts,
                        output_dir,
                        pending_page,
                        pending_table_idx
                    )

                    print(
                        f"✅ Saved merged table: "
                        f"{output_path}"
                    )

                    results.append({
                        "page": pending_page,
                        "table": pending_table_idx,
                        "output_path": output_path,
                        "expected_rows": pending_expected_rows
                    })

                    pending_parts = []
                    pending_page = None
                    pending_table_idx = None
                    pending_expected_rows = 0

                else:
 
                    print(
                        f"➡️ Table still continuing "
                        f"on Page {page_num}"
                    )
 
                    bbox = (
                        0,
                        0,
                        page.width,
                        page.height
                    )
 
                    pending_parts.append(
                        page.crop(bbox)
                    )
 
                    # A middle continuation page (tables spanning 3+
                    # pages) also contributes service rows - count them
                    # too, or they'd silently be dropped from the
                    # expected total.
                    pending_expected_rows += count_service_rows(
                        page,
                        0,
                        page.height
                    )
 
                    continue

            # =====================================
            # NO TABLES
            # =====================================

            if not start_positions and not end_positions:

                print(
                    f"❌ No tables found "
                    f"on Page {page_num}"
                )

                continue

            # =====================================
            # COMPLETE TABLES
            # =====================================

            table_count = min(
                len(start_positions),
                len(end_positions)
            )

            for idx in range(table_count):

                table_counter_on_page += 1

                start_y = start_positions[idx]
                end_y = end_positions[idx]

                bbox = (
                    0,
                    start_y,
                    page.width,
                    end_y
                )

                cropped_page = page.crop(
                    bbox
                )

                output_path = os.path.join(
                    output_dir,
                    f"page_{page_num}_table_{table_counter_on_page}.png"
                )

                cropped_page.to_image(
                    resolution=300
                ).save(output_path)

                expected_rows = count_service_rows(
                    page,
                    start_y,
                    end_y
                )

                print(
                    f"✅ Saved: {output_path}"
                    f" | Rows={expected_rows}"
                )

                results.append({
                    "page": page_num,
                    "table": table_counter_on_page,
                    "output_path": output_path,
                    "expected_rows": expected_rows
                })

            # =====================================
            # CONTINUED TABLE START
            # =====================================

            if len(start_positions) > table_count:

                table_counter_on_page += 1

                start_y = start_positions[
                    table_count
                ]

                pending_expected_rows = (
                    count_service_rows(
                        page,
                        start_y,
                        page.height
                    )
                )

                bbox = (
                    0,
                    start_y,
                    page.width,
                    page.height
                )

                pending_parts = [
                    page.crop(bbox)
                ]

                pending_page = page_num
                pending_table_idx = table_counter_on_page

                print(
                    f"➡️ Table started on "
                    f"Page {page_num} "
                    f"and continues"
                )

    return results


def _merge_and_save(parts, output_dir, page_num, table_idx):
    """Stack a list of pdfplumber crop objects vertically into one PNG."""

    images = [p.to_image(resolution=300).original for p in parts]

    total_height = sum(img.height for img in images)
    max_width = max(img.width for img in images)

    merged = Image.new("RGB", (max_width, total_height), "white")

    y_offset = 0
    for img in images:
        merged.paste(img, (0, y_offset))
        y_offset += img.height

    output_path = os.path.join(
        output_dir, f"page_{page_num}_table_{table_idx}.png"
    )
    merged.save(output_path)
    return output_path


# =========================
# TABLE ENHANCEMENT
# =========================
def make_table(image_path):
    img = cv2.imread(image_path)
    gray = cv2.imread(image_path, 0)

    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

    sums = np.sum(thresh, axis=1)
    th = (thresh.shape[1] * 255) * 0.6
    lines = np.where(sums > th)[0]

    for l in lines:
        cv2.line(img, (0, l), (thresh.shape[1], l), (0, 0, 0), 1)

    return img

def convert_amounts_to_string(obj):

    amount_fields = {
    "total_charge",
    "non_covered_charges",
    "deductible",
    "provider_discount",
    "provider_withhold",
    "remaining_covered_charges",
    "copay_coinsurance",
    "patient_responsibility",
    "total_benefit",
    "benefit_paid_to_provider"
}

    if isinstance(obj, dict):

        new_obj = {}

        for k, v in obj.items():

            if k in amount_fields:

                try:

                    clean_value = (
                        str(v)
                        .replace("$", "")
                        .replace(",", "")
                        .strip()
                    )

                    new_obj[k] = f"{float(clean_value):.2f}"

                except:
                    new_obj[k] = ""

            else:
                new_obj[k] = convert_amounts_to_string(v)

        return new_obj

    elif isinstance(obj, list):

        return [convert_amounts_to_string(i) for i in obj]

    return obj

def count_service_rows(
    page,
    region_top,
    region_bottom
):

    words = page.extract_words()

    service_rows = set()

    for w in words:

        text = w["text"].strip()

        if re.fullmatch(r"D\d{4}", text):

            y = round(
                float(w["top"]),
                1
            )

            if region_top <= y <= region_bottom:
                service_rows.add(y)

    return len(service_rows)



# =========================
# PROMPT
# =========================
def build_prompt(pdf_name):

    return """
You are extracting structured financial data from a dental EOB image.

STRICT RULES

OUTPUT ONLY VALID JSON.

DO NOT WRITE EXPLANATIONS.

DO NOT HALLUCINATE.

DO NOT CALCULATE VALUES.

DO NOT REMOVE DUPLICATE ROWS.

IF TWO ROWS LOOK IDENTICAL,
EXTRACT BOTH ROWS.

------------------------------------------------
PATIENT EXTRACTION
------------------------------------------------

patient_name
→ Extract ONLY from the value after:

Patient:

Example:

Patient: John Patrick Linner

Return:

"John Patrick Linner"

------------------------------------------------
COLUMN RULES
------------------------------------------------

date_of_service
→ Extract ONLY from the Service date column.

procedure_code
→ Extract ONLY from the Procedure code column.

total_charge
→ Extract ONLY from the Total charge column.

non_covered_charges
→ Extract ONLY from the Non-covered charges column.

deductible
→ Extract ONLY from the Deductible column.

provider_discount
→ Extract ONLY from the Provider discount/amount not covered column.

provider_withhold
→ Extract ONLY from the Provider withhold column.

remaining_covered_charges
→ Extract ONLY from the Remaining covered charges column.

copay_coinsurance
→ Extract ONLY from the Copay/coinsurance column.

patient_responsibility
→ Extract ONLY from the Patient responsibility column.

total_benefit
→ Extract ONLY from the Total benefit column.

benefit_paid_to_provider
→ Extract ONLY from the Benefit paid to provider column.


NEVER TAKE VALUES FROM NEIGHBORING COLUMNS.

NEVER SHIFT VALUES BETWEEN COLUMNS.

------------------------------------------------
SERVICE ROW RULES
------------------------------------------------

A SERVICE ROW EXISTS ONLY IF:

date_of_service exists

AND

procedure_code exists

If either is missing:

DO NOT CREATE A SERVICE ROW.

------------------------------------------------
TOTALS ROW
------------------------------------------------

Extract totals ONLY from the final row labeled:

Totals

The Totals row is NOT a service row.

DO NOT extract service date from Totals row.

DO NOT extract procedure code from Totals row.

DO NOT calculate totals.

ONLY extract values visible in the Totals row.

----------------------
totals.reason_code
→ Extract ONLY from the Reason code(s) column
of the Totals row.

Example:

Totals ........ A69

Return:

"A69"
------------------------------------------------
DUPLICATE ROW RULES
------------------------------------------------

Financial data requires exact row-level extraction.

DO NOT REMOVE DUPLICATE ROWS.

DO NOT MERGE DUPLICATE ROWS.

DO NOT DEDUPLICATE ROWS.

If two or more service rows contain identical values,
extract each occurrence as a separate service row.

Every visible service row in the table must appear in the output.

------------------------------------------------
OUTPUT
------------------------------------------------

    {
        "patient_name": {
            "value": "",
            "confidence": 0.0
        },

        "provider": {
            "value": "",
            "confidence": 0.0
        },

        "date_of_service": {
            "value": "",
            "confidence": 0.0
        },

        "services": [
            {
                "procedure_code": {
                    "value": "",
                    "confidence": 0.0
                },

                "total_charge": {
                    "value": "",
                    "confidence": 0.0
                },

                "non_covered_charges": {
                    "value": "",
                    "confidence": 0.0
                },

                "deductible": {
                    "value": "",
                    "confidence": 0.0
                },

                "provider_discount": {
                    "value": "",
                    "confidence": 0.0
                },

                "provider_withhold": {
                    "value": "",
                    "confidence": 0.0
                },

                "remaining_covered_charges": {
                    "value": "",
                    "confidence": 0.0
                },

                "copay_coinsurance": {
                    "value": "",
                    "confidence": 0.0
                },

                "patient_responsibility": {
                    "value": "",
                    "confidence": 0.0
                },

                "total_benefit": {
                    "value": "",
                    "confidence": 0.0
                },

                "benefit_paid_to_provider": {
                    "value": "",
                    "confidence": 0.0
                }
            }
        ],

        "totals": {
            "total_charge": {
                "value": "",
                "confidence": 0.0
            },

            "non_covered_charges": {
                "value": "",
                "confidence": 0.0
            },

            "deductible": {
                "value": "",
                "confidence": 0.0
            },

            "provider_discount": {
                "value": "",
                "confidence": 0.0
            },

            "provider_withhold": {
                "value": "",
                "confidence": 0.0
            },

            "remaining_covered_charges": {
                "value": "",
                "confidence": 0.0
            },

            "copay_coinsurance": {
                "value": "",
                "confidence": 0.0
            },

            "patient_responsibility": {
                "value": "",
                "confidence": 0.0
            },

            "total_benefit": {
                "value": "",
                "confidence": 0.0
            },

            "benefit_paid_to_provider": {
                "value": "",
                "confidence": 0.0
            },

            "reason_code": {
                "value": "",
                "confidence": 0.0
            }
        }
    }

    
     For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{
  "value": "",
  "confidence": ""
}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.

    ------------------------------------------------
    FINAL CHECK BEFORE RESPONDING
    ------------------------------------------------

    Before returning the JSON, verify:

    1. EVERY field contains:
       - "value"
       - "confidence"

    2. NO extracted field is a plain string.

    3. Every service row follows the exact structure above.

    4. Every totals field follows the exact structure above.

    5. The Totals row is NOT included inside services.

    6. Duplicate service rows are preserved.

    7. Rows missing date_of_service OR procedure_code are NOT created.

    8. Do NOT add fields that are not defined in the schema.

    9. Return ONLY valid JSON.
"""


def parse_amount(x):
    if x is None or x == "":
        return 0.0
    return float(
        str(x)
        .replace("$", "")
        .replace(",", "")
        .strip()
    )

def normalize_service_dates(obj):
    """
    Normalize service date formats.

    Examples:
    02/18-02/18/2026 -> 02/18/2026
    10/10-10/10/2024 -> 10/10/2024
    05/10/23-05/10/23 -> 05/10/23
    123022 -> 12/30/22
    """

    if isinstance(obj, dict):
        for k, v in obj.items():

            if k in ("service_date", "service_dates", "date_of_service") and isinstance(v, str):

                v = v.strip()

                # ----------------------------------
                # MMDDYY -> MM/DD/YY
                # Example: 123022 -> 12/30/22
                # ----------------------------------
                if re.fullmatch(r"\d{6}", v):
                    obj[k] = f"{v[:2]}/{v[2:4]}/{v[4:]}"
                    continue

                # ----------------------------------
                # Handle date ranges
                # ----------------------------------
                if "-" in v:
                    left, right = [x.strip() for x in v.split("-", 1)]

                    # MM/DD-MM/DD/YYYY -> keep second date
                    if (
                        re.fullmatch(r"\d{2}/\d{2}", left)
                        and re.fullmatch(r"\d{2}/\d{2}/\d{4}", right)
                    ):
                        obj[k] = right
                        continue

                    # MM/DD/YYYY-MM/DD/YYYY -> keep first date
                    if (
                        re.fullmatch(r"\d{2}/\d{2}/\d{4}", left)
                        and re.fullmatch(r"\d{2}/\d{2}/\d{4}", right)
                    ):
                        obj[k] = left
                        continue

                    # MM/DD/YY-MM/DD/YY -> keep first date
                    if (
                        re.fullmatch(r"\d{2}/\d{2}/\d{2}", left)
                        and re.fullmatch(r"\d{2}/\d{2}/\d{2}", right)
                    ):
                        obj[k] = left
                        continue

            else:
                normalize_service_dates(v)

    elif isinstance(obj, list):
        for item in obj:
            normalize_service_dates(item)

    return obj

def compute_totals_from_services(services):

    return {
        "total_charge": round(sum(parse_amount(s.get("total_charge","")) for s in services),2),
        "non_covered_charges": round(sum(parse_amount(s.get("non_covered_charges","")) for s in services),2),
        "deductible": round(sum(parse_amount(s.get("deductible","")) for s in services),2),
        "provider_discount": round(sum(parse_amount(s.get("provider_discount","")) for s in services),2),
        "provider_withhold": round(sum(parse_amount(s.get("provider_withhold","")) for s in services),2),
        "remaining_covered_charges": round(sum(parse_amount(s.get("remaining_covered_charges","")) for s in services),2),
        "copay_coinsurance": round(sum(parse_amount(s.get("copay_coinsurance","")) for s in services),2),
        "patient_responsibility": round(sum(parse_amount(s.get("patient_responsibility","")) for s in services),2),
        "total_benefit": round(sum(parse_amount(s.get("total_benefit","")) for s in services),2),
        "benefit_paid_to_provider": round(sum(parse_amount(s.get("benefit_paid_to_provider","")) for s in services),2)
    }


def validate_patient_totals(patient: dict, provider, patient_name: str, expected_row_count: int):

    services = patient.get("services", [])
    totals = patient.get("totals",{})
    print(f"validation for {patient_name}")
    print(f"provider name is :{provider}")

    totals_reason_code = (
    totals.get("reason_code", "")
    .strip()
    .upper()
)
    field_names = list(compute_totals_from_services([]).keys())   # ADD
    total_fields = len(field_names)   

    if not services:
        field_errors = [
            {"field": f, "computed": 0.0, "extracted": None}
            for f in field_names
        ]
        return False, "No services found", [{"error": "empty services"}] + field_errors, total_fields

    computed_totals = compute_totals_from_services(services)

    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [{patient_name}]")
    print("-" * 80)

    # ===== FIELD VALIDATION =====
    for field, computed_value in computed_totals.items():

        # ====================================
        # A69 SPECIAL RULE
        # ====================================
        if (
            field == "benefit_paid_to_provider"
            and totals_reason_code == "A69"
        ):
            extracted_value = round(
                parse_amount(totals.get(field, "")),
                2
            )

            computed_value = extracted_value

            icon = "✅"
            status = "MATCH"

            line = (
                f"{icon} {field:25s} "
                f"computed={computed_value:<10} | "
                f"extracted={extracted_value:<10} "
                f"{status} (A69 override)"
            )

            print(line)
            result_validation += "\n" + line
            continue

        extracted_value = round(
            parse_amount(totals.get(field, "")),
            2
        )

        diff = round(computed_value - extracted_value, 2)
        match = abs(diff) <= 0.01   # 🔥 tolerance fix

        if match:
            icon = "✅"
            status = "MATCH"
        else:
            icon = "❌"
            status = "MISMATCH"
            has_error = True

            errors.append({
                "type": "field_mismatch",
                "field": field,
                "computed": computed_value,
                "extracted": extracted_value,
                "difference": diff
            })

        line = f"{icon} {field:25s} computed={computed_value:<10} | extracted={extracted_value:<10} {status}"
        print(line)
        result_validation += "\n" + line

    # ===== ROW COUNT VALIDATION =====
    extracted_row_count = len(services)

    if expected_row_count == extracted_row_count:
        icon = "✅"
        status = "MATCH"
    else:
        icon = "❌"
        status = "MISMATCH"
        has_error = True

        errors.append({
            "type": "row_count_mismatch",
            "expected_rows": expected_row_count,
            "extracted_rows": extracted_row_count
        })

    line = f"{icon} {'total_record_rows':25s} computed={expected_row_count:<10} | extracted={extracted_row_count:<10} {status}"
    print(line)
    result_validation += "\n" + line

    print("-" * 80)

    # ===== FINAL STATUS =====
    if has_error:
        print(f"❌ [{patient_name}] Validation FAILED\n")
        return False, result_validation, errors, total_fields
    else:
        print(f"✅ [{patient_name}] Validation PASSED\n")
        return True, result_validation, [], total_fields

# =========================
# JSON CLEANER
# =========================
def extract_json(text):
    start = text.find("{")
    end = text.rfind("}") + 1
    return json.loads(text[start:end])

def save_json(data, output_path):
    with open(output_path, "w", encoding = "utf-8") as f:
        json.dump(data, f, indent = 2, ensure_ascii = False)

def check_claim_denied(pdf_path):

    denial_keywords = [
        "denied",
        "denial"
    ]

    with pdfplumber.open(pdf_path) as pdf:

        for page_num, page in enumerate(pdf.pages, start=1):

            full_text = page.extract_text()

            if not full_text:
                continue

            searchable_text = full_text.lower()

            for keyword in denial_keywords:

                if keyword in searchable_text:

                    print(
                        f"❌ Claim denied keyword found: "
                        f"'{keyword}' on page {page_num}"
                    )

                    return "denied"

    return "not denied"

def run_pipeline(pdf_path, output_dir="EOB_OUTPUT/Moda", company_name= "Moda"):

    pdf_name = os.path.basename(pdf_path).split(".")[0].split("_")[-1]
    pdf_full_name = os.path.basename(pdf_path)


    base_dir = os.path.join(output_dir, pdf_name)
    cropped_dir = os.path.join(base_dir, "cropped_images")

    os.makedirs(base_dir, exist_ok=True)
    os.makedirs(cropped_dir, exist_ok=True)

    image_paths = rotate_and_crop_pdf(pdf_path, output_dir=cropped_dir)

    final_prompt = build_prompt(pdf_name)
    is_denied = check_claim_denied(pdf_path)
    print(f"claim status :{is_denied}")

    results = []
    confidence_results = []   

    for idx, item in enumerate(image_paths):
        img_path = item["output_path"]
        expected_rows = item["expected_rows"]

        print(f"Processing {idx+1}/{len(image_paths)}")

        image = make_table(img_path)
        image = Image.fromarray(image).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": final_prompt}
                ]
            }
        ]

        with torch.no_grad():
            output = pipe(messages, max_new_tokens=1500, temperature= 0.0, do_sample=False)

        raw = output[0]["generated_text"]
        print("this is raw output", raw )

        if isinstance(raw, list):
            raw = raw[-1]["content"]


        try:
            parsed = extract_json(raw)
            model_confidence = calculate_model_confidence(parsed)
            parsed = _unwrap_vlm_output(parsed)
            parsed["_model_confidence"] = model_confidence

            parsed = convert_amounts_to_string(parsed)
            parsed = normalize_service_dates(parsed)
            results.append(parsed)
            parsed["_expected_rows"] = expected_rows
            print("✔ extracted")
        except Exception as e:
            print(f"❌ json failed:{e}")

        for patient in results:

            is_valid, log, errors, total_fields = validate_patient_totals(
            patient=patient,
            patient_name=patient.get("patient_name", ""),
            provider = patient.get("provider", ""),
            expected_row_count=patient.get("_expected_rows", 0)  # or your external expected count
        )

        patient["validation"] = {
            "status": is_valid,
            "errors": errors
        }

        patient["_total_fields"] = total_fields 

        confidence_results = list(results)                              # ADD — insert here
        confidence_score = calculate_eob_confidence(confidence_results)
        
    for patient in results:
        patient.pop("_expected_rows", None)
        patient.pop("_total_fields", None)
        patient.pop("_model_confidence", None)

        if "totals" in patient:
            patient["totals"].pop("reason_code", None)

    final =[
        {
        "eob_id": pdf_name,
        "file_name":pdf_full_name,
        "claim_status": is_denied,
        "payor": "Delta Dental - Moda",
        "confidence_score": confidence_score, 
        "patients": results
        }
    ]

    success_path, failed_path = save_split_output(
                                        final,
                                        company_name=company_name,
                                        pdf_name=pdf_name,
                                        pdf_path=pdf_path,
                                        cropped_dir=cropped_dir,
                                    )
                                
    print(f"\n📁 Cropped images : {cropped_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final

W0901 19:17:59.700000 3526216 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 19:17:59.715000 3526216 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/Moda/pdf/Pmt_EOP_305132004.pdf")

Page 2 Size -> Width=612.0, Height=792.0
📄 Page 2 is PORTRAIT.
🔄 Rotating pages from Page 2 onwards.

🚀 Processing Page 1
⏭️ Keeping page as-is

🚀 Processing Page 2
🔄 Rotating page by 90°

🚀 Processing Page 3
🔄 Rotating page by 90°

✅ Rotated PDF Saved: EOB_OUTPUT/Moda/305132004/cropped_images/305132004/rotated_305132004.pdf

🧾 Cropping Page 1
❌ No tables found on Page 1

🧾 Cropping Page 2


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.


✅ Saved: EOB_OUTPUT/Moda/305132004/cropped_images/305132004/page_2_table_1.png | Rows=2

🧾 Cropping Page 3
❌ No tables found on Page 3
claim status :not denied
Processing 1/1


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


this is raw output [{'role': 'user', 'content': [{'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=3301x762 at 0x7972CE34A570>}, {'type': 'text', 'text': '\nYou are extracting structured financial data from a dental EOB image.\n\nSTRICT RULES\n\nOUTPUT ONLY VALID JSON.\n\nDO NOT WRITE EXPLANATIONS.\n\nDO NOT HALLUCINATE.\n\nDO NOT CALCULATE VALUES.\n\nDO NOT REMOVE DUPLICATE ROWS.\n\nIF TWO ROWS LOOK IDENTICAL,\nEXTRACT BOTH ROWS.\n\n------------------------------------------------\nPATIENT EXTRACTION\n------------------------------------------------\n\npatient_name\n→ Extract ONLY from the value after:\n\nPatient:\n\nExample:\n\nPatient: John Patrick Linner\n\nReturn:\n\n"John Patrick Linner"\n\n------------------------------------------------\nCOLUMN RULES\n------------------------------------------------\n\ndate_of_service\n→ Extract ONLY from the Service date column.\n\nprocedure_code\n→ Extract ONLY from the Procedure code column.\n\ntotal_charge\n→ Extract ONLY from

[{'eob_id': '305132004',
  'file_name': 'Pmt_EOP_305132004.pdf',
  'claim_status': 'not denied',
  'payor': 'Delta Dental - Moda',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'Kimberly Linner',
    'provider': 'Duc Tang DDS',
    'date_of_service': '12/01/22',
    'services': [{'procedure_code': 'D0120',
      'total_charge': '65.63',
      'non_covered_charges': '0.00',
      'deductible': '0.00',
      'provider_discount': '30.63',
      'provider_withhold': '0.00',
      'remaining_covered_charges': '35.00',
      'copay_coinsurance': '0.00',
      'patient_responsibility': '0.00',
      'total_benefit': '35.00',
      'benefit_paid_to_provider': '35.00'},
     {'procedure_code': 'D1110',
      'total_charge': '117.39',
      'non_covered_charges': '0.00',
      'deductible': '0.00',
      'provider_discount': '50.39',
      'provider_withhold': '0.00',
      'remaining_covered_charges': '67.00',
      'copay_coinsurance': '0.00',
      'patient_responsibility': '0.

In [3]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/Moda/pdf/Pmt_EOP_267884082.pdf")

Page 2 Size -> Width=612.0, Height=792.0
📄 Page 2 is PORTRAIT.
🔄 Rotating pages from Page 2 onwards.

🚀 Processing Page 1
⏭️ Keeping page as-is

🚀 Processing Page 2
🔄 Rotating page by 90°

🚀 Processing Page 3
🔄 Rotating page by 90°

✅ Rotated PDF Saved: EOB_OUTPUT/Moda/267884082/cropped_images/267884082/rotated_267884082.pdf

🧾 Cropping Page 1
❌ No tables found on Page 1

🧾 Cropping Page 2


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.


✅ Saved: EOB_OUTPUT/Moda/267884082/cropped_images/267884082/page_2_table_1.png | Rows=3

🧾 Cropping Page 3
❌ No tables found on Page 3
claim status :not denied
Processing 1/1


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


this is raw output [{'role': 'user', 'content': [{'type': 'image', 'image': <PIL.Image.Image image mode=RGB size=3301x887 at 0x7C6E24BAB740>}, {'type': 'text', 'text': '\nYou are extracting structured financial data from a dental EOB image.\n\nSTRICT RULES\n\nOUTPUT ONLY VALID JSON.\n\nDO NOT WRITE EXPLANATIONS.\n\nDO NOT HALLUCINATE.\n\nDO NOT CALCULATE VALUES.\n\nDO NOT REMOVE DUPLICATE ROWS.\n\nIF TWO ROWS LOOK IDENTICAL,\nEXTRACT BOTH ROWS.\n\n------------------------------------------------\nPATIENT EXTRACTION\n------------------------------------------------\n\npatient_name\n→ Extract ONLY from the value after:\n\nPatient:\n\nExample:\n\nPatient: John Patrick Linner\n\nReturn:\n\n"John Patrick Linner"\n\n------------------------------------------------\nCOLUMN RULES\n------------------------------------------------\n\ndate_of_service\n→ Extract ONLY from the Service date column.\n\nprocedure_code\n→ Extract ONLY from the Procedure code column.\n\ntotal_charge\n→ Extract ONLY from

[{'eob_id': '267884082',
  'claim_status': 'not denied',
  'payor': 'Delta Dental - Moda',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'John Patrick Linner',
    'provider': 'Duc Tang DDS',
    'date_of_service': '08/04/22',
    'services': [{'procedure_code': 'D0120',
      'total_charge': '65.63',
      'non_covered_charges': '0.00',
      'deductible': '0.00',
      'provider_discount': '30.63',
      'provider_withhold': '0.00',
      'remaining_covered_charges': '35.00',
      'copay_coinsurance': '0.00',
      'patient_responsibility': '0.00',
      'total_benefit': '35.00',
      'benefit_paid_to_provider': '35.00'},
     {'procedure_code': 'D1110',
      'total_charge': '117.39',
      'non_covered_charges': '0.00',
      'deductible': '0.00',
      'provider_discount': '50.39',
      'provider_withhold': '0.00',
      'remaining_covered_charges': '67.00',
      'copay_coinsurance': '0.00',
      'patient_responsibility': '0.00',
      'total_benefit': '67.00',